<a href="https://colab.research.google.com/github/jazoza/cultural-data-analysis/blob/main/03_CDA_Europeana_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cultural Data Analysis

Importing and Analysing Europeana data

https://www.europeana.eu/en

In [ ]:
import os
os.environ['EUROPEANA_API_KEY'] = 'hablessallmu' # add your personal/project key

In [ ]:
!pip install pyeuropeana --no-deps
!pip install fire

In [ ]:
import pyeuropeana.apis as apis
import pyeuropeana.utils as utils

### A. Identify the object you looked at during the class

In [ ]:
result = apis.search(
    query = '*', # YOUR SEARCH TERM(S)
    qf='TYPE:"3D"', # TYPE OF MEDIA
    theme = 'art', # LEAVE BLANK '' IF YOU ARE NOT SURE
    sort = 'europeana_id',
    profile = 'rich',
    rows = 20)
df = utils.search2df(result)
df.head()

In [ ]:
#df.loc[0]['uri'] # this will give you the direct link to the 1st ([0]) item
df.columns # this will list all columns in the dataset

### B. Get a section/selection of data

In [ ]:
result = apis.search(
    query = '*',
    #qf = '(skos_concept:"http://data.europeana.eu/concept/base/48" AND TYPE:IMAGE)',
    #qf = '(COUNTRY:"United Kingdom" OR COUNTRY:"Ireland")',
    #reusability = 'open AND permission',
    #landingpage = True,
    #media = True,
    #thumbnail = True,
    #colourpalette = '#0000FF',
    #theme = 'photography',
    theme = 'industrial',
    sort = 'europeana_id',
    profile = 'rich',
    rows = 200,
    )

In [ ]:
import pandas as pd

items = result.get("items", [])
df = pd.json_normalize(items)

df.columns[:20]

In [ ]:
def flat(d):
    return list(dict.fromkeys(v for vals in d.values() for v in vals))

def get_subject(item):
    subj = flat(item.get('dcSubjectLangAware', {}))
    if subj:
        return subj, 'provider'
    c = item.get('edmConceptPrefLabelLangAware', {})
    return (c.get('en') or flat(c)), 'europeana concept'

subjects = [get_subject(it) for it in result['items']]

assert len(subjects) == len(df), f"{len(subjects)} items vs {len(df)} rows"

df['subject'] = [s for s, _ in subjects]
df['subject_source'] = [src for _, src in subjects]

In [ ]:
df.head()

### Data structure

#### Temporal structure

In [ ]:
# time extraction function
def extract_year(x):
    # 1. handle missing values safely (scalar-only check)
    if x is None:
        return "NaN"
    # 2. handle lists explicitly
    if isinstance(x, list):
        if len(x) == 0:
            return "NaN"
        x = x[0]  # take first element
    # 3. convert to string safely
    x = str(x)
    # 4. extract year (first 4 digits if possible)
    digits = "".join(ch for ch in x[:10] if ch.isdigit())
    if len(digits) >= 4:
        return int(digits[:4])
    return "NaN"

In [ ]:
df["year"] = pd.to_numeric(df["edmTimespan"].apply(extract_year), errors="coerce")

In [ ]:
from collections import Counter
import pandas as pd

# timespan analysis: relevant column is 'edmTimespan'
columns = 'edmTimespan'

# global counter
year_counts = Counter()

# Apply the extraction directly to the in-memory dataframe `df`
if columns in df.columns:
    years = df[columns].apply(extract_year)
    year_counts.update(years)
else:
    print(f"Column '{columns}' not found in the DataFrame.")

# Convert all keys to strings to prevent mixed-type comparison during sorting
year_counts = {str(k): v for k, v in year_counts.items()}
year_counts = dict(sorted(year_counts.items()))
dict(list(year_counts.items())[:5])

In [ ]:
import re
import matplotlib.pyplot as plt

col = 'edmTimespan'

def years_in(cell):
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return []
    text = ' '.join(map(str, cell)) if isinstance(cell, (list, tuple)) else str(cell)
    return [int(y) for y in re.findall(r'(?<!\d)(1[0-9]{3}|20[0-2][0-9])(?!\d)', text)]

year_series = (df[col].apply(years_in).explode().dropna().astype(int)
               .value_counts().sort_index())

# one bar per year, including years with zero items
year_series = year_series.reindex(
    range(year_series.index.min(), year_series.index.max() + 1), fill_value=0)

fig, ax = plt.subplots(figsize=(36, 6))
ax.bar(year_series.index, year_series.values, width=0.9, color='grey')
ax.set_xticks(range(year_series.index.min(), year_series.index.max() + 1, 5))
ax.tick_params(axis='x', rotation=90)
ax.set_xlabel('Year')
ax.set_ylabel('Count')
ax.set_title('Distribution of edmTimespan by Year')
ax.margins(x=0.01)
plt.tight_layout()
plt.savefig('edmTimespan_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# define time periods
bins = [0, 1491, 1789, 1903, 1929, 1955, float("inf")]

labels = [
    "0–1491",
    "1492–1789",
    "1790–1903",
    "1904–1929",
    "1930–1955",
    "1956–today"
]

df["period"] = pd.cut(
    df["year"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

colors = [
    "#C1BEDB",
    "#A7A2CE",
    "#8E86C1",
    "#746AB4",
    "#5A4EA7",
    "#463D86"
]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["period"] = pd.cut(df["year"], bins=bins, labels=labels, include_lowest=True)

counts = df["period"].value_counts().reindex(labels, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(counts.index, counts.values, color=colors)
ax.bar_label(bars, padding=3)
ax.set_xlabel("Period")
ax.set_ylabel("Count")
ax.set_title("Items by time period")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("period_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Undated items excluded: {df['year'].isna().sum()}")

#### Subjects

In [ ]:
# count the most frequent subjects
from collections import Counter

def extract_subjects(items):
    """One list of subjects per item: provider subjects first, Europeana concept labels as fallback."""
    out = []
    for it in items:
        subj = [v for vals in it.get('dcSubjectLangAware', {}).values() for v in vals]
        if not subj:
            c = it.get('edmConceptPrefLabelLangAware', {})
            subj = c.get('en') or [v for vals in c.values() for v in vals]
        # split "a; b; c" strings, strip whitespace, drop URIs and duplicates
        subj = [s.strip() for v in subj for s in v.split(';')
                if s.strip() and not s.strip().startswith('http')]
        out.append(list(dict.fromkeys(subj)))
    return out

df['subject'] = extract_subjects(result['items'])   # assumes df rows match result['items'] order

subject_counts = Counter(s for subs in df['subject'] for s in subs)
all_subjects = sorted(subject_counts)               # every distinct subject

print(f"{len(all_subjects)} distinct subjects; {(df['subject'].str.len() == 0).sum()} items without any")
subject_counts.most_common(30)

#### Descriptions

In [ ]:
import re
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def top_terms(texts, n=30, min_len=3, stop=ENGLISH_STOP_WORDS):
    words = (w for t in texts for w in re.findall(r"[^\W\d_]+", t.lower()))
    return Counter(w for w in words if len(w) >= min_len and w not in stop).most_common(n)

# English descriptions only, since the stopword list is English
df['descriptions_en'] = extract_descriptions(result['items'], lang='en')
texts_en = [d for ds in df['descriptions_en'] for d in ds]

top_terms(texts_en)